# Eksperimen Elitism pada Genetic Algorithm untuk Penjadwalan Kuliah

## Non-Elitism vs Static Elitism vs Dynamic Elitism

Studi kasus: **optimasi jadwal perkuliahan** menggunakan *Genetic Algorithm* (GA).
Fokus utama assignment ini adalah operator **ELITISM**: bagaimana kita mempertahankan solusi terbaik
dari generasi ke generasi, dan bagaimana perilaku tersebut berubah ketika **ukuran elitism tidak
statis**, melainkan dijadwalkan secara adaptif.

### Latar belakang ide

Dalam *deep learning*, laju pembelajaran (*learning rate*) tidak selalu konstan.
Scheduler seperti `ReduceLROnPlateau` bekerja dengan logika:

- jika metrik membaik → pertahankan laju,
- jika metrik stagnan selama `patience` generasi → perkecil laju agar eksploitasi lebih halus,
- jika metrik memburuk → langsung perbaiki laju secara signifikan.

Konsep yang sama coba diterapkan pada elitism:

| Deep learning | Eksperimen ini |
|---|---|
| loss / validation metric | best fitness populasi |
| learning rate | **ukuran elitism** (`elite_size`) |
| scheduler patience | `SCHEDULER_PATIENCE` |
| cooldown | `SCHEDULER_COOLDOWN` |
| early stopping | `EARLY_STOPPING_PATIENCE` |
| gradient clipping ( safeguard ) | `MIN_DIVERSITY` + `MIN/MAX_ELITISM` + global best |

Jadi **dynamic elitism** = *elitism dengan scheduler*, bukan elitism yang terus naik monoton.

### Rumusan masalah dan hipotesis

**Pertanyaan utama:** apakah dynamic elitism berbahaya, mengingat bahwa elitism yang makin tinggi
meny|population diversity| makin kecil?

**Hipotesis yang diuji:**

1. Tanpa elitism, best fitness dapat turun antar-generasi (solusi terbaik hilang).
2. Dengan elitism, best fitness bersifat monoton naik.
3. Static elitism menjaga eksplorasi (diversity tinggi) tetapi lambat keluar dari *stagnation*.
4. Dynamic elitism keluar dari *stagnation* lebih cepat (rata-rata fitness naik lebih tinggi),
   tetapi membayar dengan diversity yang lebih rendah.
5. Elitism yang dibiarkan naik tanpa batas memang berbahaya: diversity anjlok dan
   best fitness justru turun. Karena itu scheduler wajib dibatasi oleh
   `MIN_ELITISM`, `MAX_ELITISM`, `MIN_DIVERSITY`, `cooldown`, dan penyimpanan **global best**.

### Temuan Eksperimen

Semua angka di bawah dihasilkan oleh notebook ini: satu run utama (`SEED = 42`) untuk grafik
dan tabel per generasi, lalu 10 run (seed 0-9) untuk pemodelan multi-run dan ablasi.

**1. Tanpa elitism, solusi terbaik benar-benar hilang.**
Pada seed utama, non-elitism berjalan 105 generasi, best-ever hanya **70**, kurva best
fitness turun **39 kali**, dan rata-rata populasi berakhir di **2.00**. Pada 10 run,
rata-rata best-ever **79.0** dan **0 dari 10** run mencapai jadwal tanpa pelanggaran.

**2. Elitism membuat best fitness monoton.**
Static maupun dynamic elitism mencatat **0 penurunan** best fitness pada seluruh 10 run,
sesuai $BestFitness_{t+1} \ge BestFitness_t$.

**3. Dynamic lebih cepat sampai optimum, tetapi membayar dengan diversity.**

| Metrik (rata-rata 10 run) | Non-Elitism | Static Elitism | Dynamic Elitism |
|---|---:|---:|---:|
| Best-ever fitness | 79.0 ± 9.4 | 97.0 ± 4.8 | **99.0 ± 2.1** |
| Run mencapai fitness 100 | 0 / 10 | 6 / 10 | **8 / 10** |
| Penurunan best fitness | 30.6 | 0.0 | 0.0 |
| Rata-rata fitness sepanjang run | 3.56 | 28.36 | **51.80** |
| Diversity akhir | 1.00 | 0.92 | 0.71 |
| Diversity terendah | 0.95 | 0.80 | **0.48** |
| Elitism maksimum | 0 | 2 | 12 |
| Rata-rata generasi ke optimum | tidak tercapai | 40.5 | 46.6 |

Pada seed utama, dynamic mencapai fitness 100 di **generasi 28**, static di **generasi 48**,
dan dynamic menutup run dengan rata-rata populasi **71.50** melawan **39.33** untuk static.
Jadi dynamic memang lebih eksploitatif; keunggulannya bukan pada "*mencapai* lebih tinggi"
melainkan *memakai* solusi yang sudah ditemukan dengan lebih konsisten.

**4. Elitism besar memang memotong diversity.**
Dalam run dynamic, korelasi Pearson `elite_size` terhadap rasio diversity = **-0.664**.
Regresinya memberi -2.8 poin persen rasio diversity untuk setiap tambahan satu elite.

**5. Pengaman scheduler terbukti bekerja, dan memang dibutuhkan.**

| Varian | Best-ever | Diversity akhir | Diversity terendah | Elitism maks | Naik / turun elitism |
|---|---:|---:|---:|---:|---:|
| A — dengan pengaman | 99.0 | **0.71** | **0.48** | 12 (40%) | 11.6 / 8.6 |
| B — tanpa lantai diversity | 99.5 | 0.57 | 0.39 | 12 (40%) | 4.0 / 0.0 |
| C — tanpa batas atas | 97.5 | 0.27 | 0.17 | 26 (87%) | 6.0 / 0.0 |

Pada seed utama lantai diversity menyala **7 kali** dan setiap kali menurunkan elitism
kembali dari 12 ke 8, tepat sebelum populasi terlalu homogen. Varian C membiarkan elite
tumbuh sampai 87% populasi, populasinya nyaris anjlok (diversity 0.17), dan justru
best-ever-nya turun ke 97.5. Varian B sedikit lebih tinggi di puncak (99.5), tetapi
populasinya lebih rapuh karena scheduler tidak pernah bereaksi.

**Kesimpulan satu kalimat:** dynamic elitism bukan lebih baik secara mutlak; ia lebih cepat
dan lebih sering mencapai optimum **tetapi hanya selama diversity dijaga**. Begitu pengaman
dilepas, eksplorasi hilang dan hasil terbaik ikut turun.

---

> Muhammad Fikri
> 24060124130069

## 1. Studi Kasus

Program Studi Informatika ingin menyusun jadwal kuliah dengan memperhatikan:

### Hard Constraint

Pelanggaran hard constraint dianggap serius.

- Seorang dosen tidak boleh mengajar dua mata kuliah pada waktu yang sama.
- Satu ruang tidak boleh digunakan dua mata kuliah pada waktu yang sama.
- Kapasitas ruang harus cukup untuk jumlah mahasiswa.
- Mata kuliah pada semester yang sama tidak boleh berlangsung pada waktu yang sama.

### Soft Constraint

Soft constraint boleh dilanggar, tetapi mengurangi kualitas jadwal.

- Sebisa mungkin jadwal mengikuti slot waktu yang disukai dosen.

### Bobot penalti

| Pelanggaran | Penalti |
|---|---:|
| Bentrok dosen | 25 |
| Bentrok ruang | 25 |
| Kapasitas ruang kurang | 20 |
| Bentrok semester | 20 |
| Tidak sesuai preferensi dosen | 5 |

Fitness dihitung dengan:

$$
Fitness = \max(0,\ 100 - TotalPenalty)
$$

Semakin tinggi fitness, semakin baik jadwal. Karena `fitness` tidak pernah negatif,
nilai **100** adalah batas atas yang diketahui secara analitis: jadwal tanpa pelanggaran sama sekali.
Nilai inilah yang dipakai sebagai kondisi *stop* pada bagian eksperimen.

In [ ]:
import copy
import math
import random
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

# =====================================================================
# KONFIGURASI EKSPERIMEN
# Semua parameter perilaku elitism dikumpulkan di sini agar mudah
# ditelusuri dan diubah saat melakukan ablation.
# =====================================================================

# --- Reproduibilitas ---
SEED = 42
N_RUNS = 10                 # jumlah seed untuk eksperimen multi-run

# --- Struktur populasi ---
POP_SIZE = 30               # ukuran populasi
MAX_GENERATIONS = 200       # batas aman atas jumlah generasi

# --- Operator genetika (identik untuk ketiga skema) ---
MUTATION_RATE = 0.12        # peluang tiap gene dimutasi
CROSSOVER_POINTS = 2        # crossover 2 titik
TOURNAMENT_SIZE = 3         # jumlah peserta tournament selection

# --- Elitism: skema eksperimen ---
STATIC_ELITISM = 2          # elite_size pada skema static elitism

# --- Elitism: scheduler dinamis ---
INITIAL_ELITISM = 2         # elitism pada generasi pertama
MIN_ELITISM = 1             # batas bawah elitism
MAX_ELITISM = 12            # batas atas elitism (≈ 40% dari populasi)
ELITISM_FACTOR = 1.5        # faktor geometris, meniru gamma pada ReduceLROnPlateau
MIN_DIVERSITY = 0.60        # lantai diversity (rasio individu unik / ukuran populasi)

# --- Scheduler & early stopping ---
SCHEDULER_PATIENCE = 5      # stagnan berapa generasi sebelum elitism dinaikkan
SCHEDULER_COOLDOWN = 4      # jeda antar perubahan elitism (mencegah osilasi)
EARLY_STOPPING_PATIENCE = 50   # hentikan run jika global best tidak membaik selama ini
# Run SELALU diteruskan sampai early stopping (atau batas generasi). Reaching optimum
# tidak menghentikan run, hanya dicatat pada `first_optimum_generation` supaya kita bisa
# mengukur "berapa lama elitism dibutuhkan" tanpa memotong kurva.
# Set True hanya bila ingin melihat kurva berhenti tepat di fitness 100.
STOP_AT_OPTIMUM = False

# --- Tampilan tabel ---
SHOW_FULL_TABLE = False     # True = tampilkan seluruh tabel tiap generasi

# Nama & warna tiap skema agar konsisten di semua grafik
SCHEME_LABELS = {
    "none": "Non-Elitism",
    "static": "Static Elitism",
    "dynamic": "Dynamic Elitism",
}
SCHEME_COLORS = {
    "none": "#d62728",
    "static": "#1f77b4",
    "dynamic": "#2ca02c",
}

print("Environment siap.")
print(f"Populasi           : {POP_SIZE}")
print(f"Elitism statis     : {STATIC_ELITISM}")
print(f"Elitism dinamis    : {INITIAL_ELITISM} -> [{MIN_ELITISM}, {MAX_ELITISM}] (faktor {ELITISM_FACTOR})")
print(f"Lantai diversity   : {MIN_DIVERSITY:.0%}")
print(f"Scheduler patience : {SCHEDULER_PATIENCE} (cooldown {SCHEDULER_COOLDOWN})")
print(f"Early stopping     : {EARLY_STOPPING_PATIENCE} generasi tanpa perbaikan")

In [ ]:
# Data mata kuliah
courses = [
    {"name": "Algoritma",       "lecturer": "Andi",  "students": 35, "semester": 3},
    {"name": "Basis Data",      "lecturer": "Budi",  "students": 40, "semester": 3},
    {"name": "RPL",             "lecturer": "Andi",  "students": 30, "semester": 5},
    {"name": "Kecerdasan AI",   "lecturer": "Citra", "students": 45, "semester": 5},
    {"name": "Jaringan",        "lecturer": "Budi",  "students": 25, "semester": 5},
    {"name": "Data Mining",     "lecturer": "Citra", "students": 38, "semester": 5},
    {"name": "Sistem Operasi",  "lecturer": "Deni",  "students": 42, "semester": 3},
    {"name": "IoT",             "lecturer": "Deni",  "students": 28, "semester": 7},
]

rooms = [
    {"name": "R101", "capacity": 30},
    {"name": "R102", "capacity": 40},
    {"name": "R103", "capacity": 50},
]

timeslots = [
    "Senin 08.00",
    "Senin 10.00",
    "Selasa 08.00",
    "Selasa 10.00",
    "Rabu 08.00",
    "Rabu 10.00",
]

# Preferensi dosen hanya untuk contoh soft constraint
lecturer_preferences = {
    "Andi":  {0, 1, 2},
    "Budi":  {1, 2, 3},
    "Citra": {2, 3, 4},
    "Deni":  {0, 4, 5},
}

formatted_lecturer_preferences = {}
for lecturer, slot_ids in lecturer_preferences.items():
    preferred_times = [timeslots[slot_id] for slot_id in sorted(list(slot_ids))]
    formatted_lecturer_preferences[lecturer] = ", ".join(preferred_times)

df_courses = pd.DataFrame(courses)
df_rooms = pd.DataFrame(rooms)

print("DATA MATA KULIAH")
display(df_courses)

print("\nDATA RUANG")
display(df_rooms)

print("\nSLOT WAKTU")
display(pd.DataFrame({"slot_id": range(len(timeslots)), "waktu": timeslots}))

print("\nPREFERENSI DOSEN")
df_lecturer_preferences = pd.DataFrame.from_dict(
    formatted_lecturer_preferences,
    orient="index",
    columns=["Preferred Timeslots"]
)
display(df_lecturer_preferences)

## 2. Representasi Kromosom

Satu **individu** mewakili satu jadwal lengkap.

Satu **gene** menyimpan:

$$
Gene_i = (Timeslot_i,\ Room_i)
$$

Contoh:

```text
[(0,1), (2,2), (1,0), ...]
```

Artinya mata kuliah pertama ditempatkan pada slot 0 (Senin 08.00) di ruang 1 (R101),
mata kuliah kedua pada slot 2 (Selasa 08.00) di ruang 2 (R102), dan seterusnya.

Dengan demikian:

```text
Chromosome = [G1][G2][G3][G4][G5][G6][G7][G8]
                 satu jadwal lengkap
```

Konsekuensi penting untuk eksperimen ini: **fitness hanya bergantung pada posisi slot dan ruang**.
Tidak adaRepresentasi waktu absolut per individu, sehingga individu dengan kromosom identik
adalah individu yang identik secara fungsional. Hal ini dipakai untuk mengukur *diversity*.

In [ ]:
def create_individual(rng=random):
    """Satu individu = daftar gene (timeslot, ruang) untuk setiap mata kuliah."""
    return [
        [rng.randrange(len(timeslots)), rng.randrange(len(rooms))]
        for _ in courses
    ]

sample_individual = create_individual(random.Random(SEED))
sample_individual

In [ ]:
# Visualisasi kromosom
fig, ax = plt.subplots(figsize=(12, 2.7))
ax.set_xlim(0, len(courses))
ax.set_ylim(0, 1)
ax.axis("off")

for i, (course, gene) in enumerate(zip(courses, sample_individual)):
    rect = plt.Rectangle((i + 0.05, 0.25), 0.9, 0.5, fill=False, linewidth=1.5)
    ax.add_patch(rect)
    ax.text(i + 0.5, 0.58, f"G{i+1}", ha="center", va="center", fontweight="bold")
    ax.text(i + 0.5, 0.42, f"S{gene[0]} R{gene[1]}", ha="center", va="center")
    ax.text(i + 0.5, 0.12, course["name"], ha="center", va="center", rotation=25, fontsize=8)

ax.set_title("Representasi Kromosom: Setiap Gene = (Slot, Ruang)")
plt.tight_layout()
plt.show()

## 3. Fitness Function dan Penalty

Fitness berfungsi memberi nilai pada setiap jadwal. Kita tidak hanya bertanya:

> "Apakah jadwal ini valid?"

Tetapi:

> "Seberapa baik jadwal ini dibandingkan jadwal lain?"

Karena itu setiap konflik diberi penalti, dan GA berusaha **mengurangi penalti dari generasi ke generasi**.

Fungsi `evaluate` mengembalikan dua jenis informasi:

- angka (`fitness`, `penalty`, `breakdown`) yang dipakai di dalam loop evolusi,
- daftar `violations` berbentuk kalimat yang dipakai untuk laporan pada bagian akhir.

In [ ]:
PENALTIES = {
    "lecturer_conflict": 25,
    "room_conflict": 25,
    "capacity": 20,
    "semester_conflict": 20,
    "preference": 5,
}


def evaluate(individual, detail=True):
    """Hitung fitness satu individu.

    detail=False dipakai di dalam loop evolusi supaya tidak membangun string
    yang tidak diperlukan (hanya angka yang dibutuhkan scheduler).
    """
    penalty = 0
    breakdown = Counter()
    violations = [] if detail else None

    # Kapasitas ruang dan preferensi dosen
    for i, course in enumerate(courses):
        slot_i, room_i = individual[i]

        if rooms[room_i]["capacity"] < course["students"]:
            penalty += PENALTIES["capacity"]
            breakdown["Kapasitas ruang"] += PENALTIES["capacity"]
            if detail:
                violations.append(
                    f"Kapasitas: {course['name']} ({course['students']} mhs) "
                    f"tidak muat di {rooms[room_i]['name']} ({rooms[room_i]['capacity']})"
                )

        if slot_i not in lecturer_preferences[course["lecturer"]]:
            penalty += PENALTIES["preference"]
            breakdown["Preferensi dosen"] += PENALTIES["preference"]
            if detail:
                violations.append(
                    f"Preferensi: {course['lecturer']} kurang menyukai slot {timeslots[slot_i]} "
                    f"untuk {course['name']}"
                )

    # Konflik antar pasangan mata kuliah
    for i in range(len(courses)):
        slot_i, room_i = individual[i]

        for j in range(i + 1, len(courses)):
            slot_j, room_j = individual[j]

            if slot_i != slot_j:
                continue

            if courses[i]["lecturer"] == courses[j]["lecturer"]:
                penalty += PENALTIES["lecturer_conflict"]
                breakdown["Bentrok dosen"] += PENALTIES["lecturer_conflict"]
                if detail:
                    violations.append(
                        f"Dosen: {courses[i]['lecturer']} mengajar {courses[i]['name']} dan "
                        f"{courses[j]['name']} pada {timeslots[slot_i]}"
                    )

            if room_i == room_j:
                penalty += PENALTIES["room_conflict"]
                breakdown["Bentrok ruang"] += PENALTIES["room_conflict"]
                if detail:
                    violations.append(
                        f"Ruang: {rooms[room_i]['name']} dipakai {courses[i]['name']} dan "
                        f"{courses[j]['name']} pada {timeslots[slot_i]}"
                    )

            if courses[i]["semester"] == courses[j]["semester"]:
                penalty += PENALTIES["semester_conflict"]
                breakdown["Bentrok semester"] += PENALTIES["semester_conflict"]
                if detail:
                    violations.append(
                        f"Semester: semester {courses[i]['semester']} memiliki {courses[i]['name']} "
                        f"dan {courses[j]['name']} pada {timeslots[slot_i]}"
                    )

    return {
        "fitness": max(0, 100 - penalty),
        "penalty": penalty,
        "breakdown": breakdown,
        "violations": violations,
    }


sample_eval = evaluate(sample_individual)
print("fitness :", sample_eval["fitness"])
print("penalty :", sample_eval["penalty"])
print("pelanggaran:")
for v in sample_eval["violations"]:
    print(" -", v)

In [ ]:
# Visualisasi breakdown penalti individu contoh
breakdown = sample_eval["breakdown"]

if breakdown:
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(list(breakdown.keys()), list(breakdown.values()), color="#4c72b0")
    ax.set_title("Breakdown Penalti pada Satu Individu")
    ax.set_ylabel("Total Penalti")
    ax.tick_params(axis="x", rotation=20)
    plt.tight_layout()
    plt.show()
else:
    print("Individu contoh kebetulan tidak memiliki penalti.")

## 4. Populasi Awal

GA tidak bekerja hanya dengan satu solusi, melainkan dengan sekumpulan solusi yang disebut
**population**:

```text
Population
├── Individu 1 → Fitness 40
├── Individu 2 → Fitness 65
├── Individu 3 → Fitness 80
├── Individu 4 → Fitness 55
└── ...
```

Agar ketiga skema dibandingkan secara adil, **populasi awal dibuat dari seed yang sama**
dan dipakai ulang pada setiap skema. Selain populasi awal, `seed` operator genetika juga sama,
sehingga satu-satunya variabel bebas yang berubah adalah skema elitism.

In [ ]:
def population_diversity(population):
    """Jumlah kromosom unik dalam populasi (proxy population diversity)."""
    return len({individual_key(ind) for ind in population})


def individual_key(individual):
    """Kunci pembanding dua individu: sama berarti individu identik secara fungsional."""
    return tuple(tuple(gene) for gene in individual)


# Satu populasi awal yang dipakai bersama oleh ketiga skema
rng_init = random.Random(SEED)
initial_population = [create_individual(rng_init) for _ in range(POP_SIZE)]
initial_fitness = [evaluate(ind, detail=False)["fitness"] for ind in initial_population]

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(initial_fitness, bins=range(0, 105, 10), color="#4c72b0", edgecolor="white")
ax.set_title("Distribusi Fitness Populasi Awal")
ax.set_xlabel("Fitness")
ax.set_ylabel("Jumlah Individu")
plt.tight_layout()
plt.show()

print("Jumlah individu        :", len(initial_population))
print("Kromosom unik (awal)   :", population_diversity(initial_population))
print("Best fitness awal      :", max(initial_fitness))
print("Average fitness awal   :", round(float(np.mean(initial_fitness)), 2))

## 5. Selection: Tournament Selection

Digunakan **tournament selection** dengan ukuran tournament `k`. `k` individu dipilih acak,
dan yang paling Fitness-nya tinggi menjadi parent:

```text
k = 3
Candidate A → Fitness 65
Candidate B → Fitness 85
Candidate C → Fitness 40
                  ↓
         B menjadi parent
```

- `k` kecil → tekanan seleksi lemah (banyak eksplorasi),
- `k` besar → tekanan seleksi kuat (banyak eksploitasi).

Nilai `k` dijaga konstan di semua skema, sehingga tidak menjadi faktor yang memengaruhi
perbandingan elitism.

In [ ]:
def selection(population, fitness_values, tournament_size=TOURNAMENT_SIZE, rng=random):
    """Tournament selection berdasarkan fitness yang sudah dihitung sebelumnya."""
    k = max(2, min(tournament_size, len(population)))
    candidates = rng.sample(range(len(population)), k)
    winner = max(candidates, key=lambda i: fitness_values[i])
    return copy.deepcopy(population[winner])

## 6. Crossover

Crossover menggabungkan bagian dari dua parent:

```text
Parent A : [A][B][C][D] | [E][F][G][H]
Parent B : [a][b][c][d] | [e][f][g][h]

                         ↓ crossover 2 titik

Child    : [A][B][C][D] | [e][f][G][H]
```

Tujuannya menggabungkan bagian jadwal yang baik dari dua solusi berbeda.

In [ ]:
def crossover(parent1, parent2, rng=random):
    """Crossover 2 titik pada level gene (per mata kuliah)."""
    size = len(courses)

    if CROSSOVER_POINTS >= 2:
        point1 = rng.randint(1, size - 2)
        point2 = rng.randint(point1 + 1, size - 1)
        child = parent1[:point1] + parent2[point1:point2] + parent1[point2:]
        return copy.deepcopy(child), (point1, point2)

    point = rng.randint(1, size - 1)
    child = parent1[:point] + parent2[point:]
    return copy.deepcopy(child), point


parent_a = selection(initial_population, initial_fitness, rng=random.Random(SEED))
parent_b = selection(initial_population, initial_fitness, rng=random.Random(SEED))
child_demo, crossover_points = crossover(parent_a, parent_b, rng=random.Random(SEED))

print("Titik crossover:", crossover_points)

In [ ]:
# Visualisasi crossover
fig, axes = plt.subplots(3, 1, figsize=(12, 5), sharex=True)

for ax, (title, individual) in zip(
    axes,
    [("Parent A", parent_a), ("Parent B", parent_b), ("Child", child_demo)],
):
    ax.set_xlim(0, len(courses))
    ax.set_ylim(0, 1)
    ax.axis("off")
    ax.set_ylabel(title, rotation=0, labelpad=45, va="center")

    for i, gene in enumerate(individual):
        ax.add_patch(plt.Rectangle((i + 0.05, 0.2), 0.9, 0.6, fill=False))
        ax.text(i + 0.5, 0.5, f"S{gene[0]} R{gene[1]}", ha="center", va="center")

    for point in np.atleast_1d(crossover_points):
        ax.axvline(point, linestyle="--", color="#d62728")

axes[-1].set_xticks(np.arange(len(courses)) + 0.5)
axes[-1].set_xticklabels([c["name"] for c in courses], rotation=25, ha="right")
fig.suptitle("Visualisasi Two-Point Crossover")
plt.tight_layout()
plt.show()

## 7. Mutation

Mutation mengubah sebagian kecil gene secara acak:

```text
Sebelum mutation:
AI → Selasa 08.00 → R102

Sesudah mutation:
AI → Rabu 08.00 → R103
```

Mutation adalah sumber utama **eksplorasi**. Ada satu keputusan metodologis yang penting
untuk validitas eksperimen ini: mutasi harus ** stokastik** — hasilnya diterima maupun tidak,
tergantung pada receptive selection di generasi berikutnya.

Jika mutasi diam-diam hanya diterima bila fitness membaik, operator tersebut bukan lagi mutasi
GA, melainkan *hill climbing*. Akibatnya elitism menjadi tidak penting, dan tujuan eksperimen
ini hilang. Maka di sini mutasi murni acak.

In [ ]:
def mutation(individual, mutation_rate=MUTATION_RATE, rng=random):
    """Mutasi acak per gene: ganti timeslot dan/atau ruang dengan peluang mutation_rate."""
    child = copy.deepcopy(individual)
    mutated_positions = []

    for i in range(len(child)):
        if rng.random() < mutation_rate:
            child[i] = [
                rng.randrange(len(timeslots)),
                rng.randrange(len(rooms)),
            ]
            mutated_positions.append(i)

    return child, mutated_positions


mutated_child, mutated_positions = mutation(child_demo, mutation_rate=0.35, rng=random.Random(SEED))
print("Gene yang berubah:", mutated_positions)

In [ ]:
# Visualisasi mutation
fig, axes = plt.subplots(2, 1, figsize=(12, 3.8), sharex=True)

for ax, title, individual in [
    (axes[0], "Sebelum Mutation", child_demo),
    (axes[1], "Sesudah Mutation", mutated_child),
]:
    ax.set_xlim(0, len(courses))
    ax.set_ylim(0, 1)
    ax.axis("off")
    ax.set_ylabel(title, rotation=0, labelpad=60, va="center")

    for i, gene in enumerate(individual):
        linewidth = 3 if (title == "Sesudah Mutation" and i in mutated_positions) else 1
        ax.add_patch(plt.Rectangle((i + 0.05, 0.2), 0.9, 0.6, fill=False, linewidth=linewidth))
        ax.text(i + 0.5, 0.5, f"S{gene[0]} R{gene[1]}", ha="center", va="center")

axes[-1].set_xticks(np.arange(len(courses)) + 0.5)
axes[-1].set_xticklabels([c["name"] for c in courses], rotation=25, ha="right")
fig.suptitle("Visualisasi Mutation: Kotak Tebal Menandai Gene yang Berubah")
plt.tight_layout()
plt.show()

## 8. Fokus Utama: Elitism

### 8.1 Apa itu Elitism?

**Elitism** adalah strategi untuk membawa beberapa individu terbaik dari generasi saat ini
**langsung** ke generasi berikutnya. Individu elite:

- tidak melalui selection,
- tidak melalui crossover,
- tidak melalui mutation,
- tidak diubah sama sekali.

Jika `POP_SIZE = 30` dan `elite_size = 2`, maka generasi baru berisi:

```text
New Population
├── Elite 1      ← dipertahankan apa adanya
├── Elite 2      ← dipertahankan apa adanya
├── Child 1      ← selection + crossover + mutation
├── ...
└── Child 28
```

Secara matematis:

$$
30 = 2\ Elite + 28\ Offspring
$$

### 8.2 Tiga skema yang dibandingkan

| Skema | `elite_size` | Perilaku |
|---|---|---|
| **Non-Elitism** | `0` | seluruh populasi diganti offspring |
| **Static Elitism** | konstan `2` | 2 terbaik disalin setiap generasi |
| **Dynamic Elitism** | berubah `2 → [1, 12]` | dijadwalkan oleh scheduler |

Ketiganya memakai populasi awal, `seed`, `mutation_rate`, `crossover`, dan `selection` yang sama.

### 8.3 Apakah dynamic elitism berbahaya?

Ya, **jika tidak dibatasi**. Perhatikan arah hubungan berikut:

```text
elitism naik  →  jumlah slot untuk individu baru mengecil
              →  population diversity turun
              →  eksplorasi melemah
              →  risiko premature convergence naik
              →  best fitness bisa ikut turun
```

Yang membuat scheduler seumpama *ReduceLROnPlateau* berbeda dari "elite yang terus naik" adalah
lima **pengaman (guardrail)** yang dipakai pada notebook ini:

1. `MAX_ELITISM` — plafon keras. Elite tidak boleh melebihi sebagian populasi.
2. `MIN_ELITISM` — lantai keras, elitism tidak pernah nol pada skema dinamis.
3. `MIN_DIVERSITY` — **lantai diversity**. Bila rasio individu unik turun di bawah ambang ini,
   scheduler justru **menurunkan** elitism, bukan menaikkannya.
4. `SCHEDULER_COOLDOWN` — jeda setelah setiap perubahan, agar scheduler tidak naik-turun
   setiap generasi (osilasi).
5. **Global best tersimpan** — satu individu terbaik yang pernah ditemukan selalu ikut
   dibawa ke generasi berikutnya, **meskipun persentase elitism diturunkan**.
   Ini menjawab kekhawatiran "kalau elitism diturunkan, bukankah solusi terbaik hilang?":
   tidak, karena solusi terbaik tidak bergantung pada rasio elitism, melainkan pada fakta
   bahwa individu itu disalin utuh ke generasi berikutnya.

Aturan main scheduler (meniru `ReduceLROnPlateau`):

```text
setiap generasi:
    jika ada improvement global best      -> reset patience, tidak ada perubahan
    jika diversity < MIN_DIVERSITY        -> turunkan elitism   (eksplorasi dipulihkan)
    jika cooldown > 0                     -> tidak ada perubahan
    jika patience >= SCHEDULER_PATIENCE   -> naikkan elitism    (eksploitasi diperkuat)
    selainnya                             -> elitism tetap
```

Perhatikan urutannya: **lantai diversity menang prioritas**. Pengaman yang bersifat korektif
tidak menghormati cooldown, sedangkan kenaikan elitism menghormatinya.
Alasannya sederhana: koreksi keselamatan tidak boleh ditunda, sedangkan dorongan tambahan
tidak perlu buru-buru.

> Catatan jujur: menaikkan elitism selalu menjadi pengorbanan eksplorasi, jadi selalu
> merupakan **kompromi**. Eksperimen ini tidak mengklaim dynamic elitism selalu mengalahkan
> static elitism; tujuannya mengukur **kapan** kompromi itu menguntungkan dan kapan ia merusak.

In [ ]:
def select_elites(scored_population, elite_size):
    """Ambil elite_size individu terbaik dari populasi yang sudah terurut.

    scored_population = [(individu, hasil_evaluate), ...] sudah terurut menurun.
    """
    if elite_size <= 0:
        return []
    return [copy.deepcopy(ind) for ind, _ in scored_population[:elite_size]]


def build_elite_pool(scored_population, elite_size, global_best, keep_global_best):
    """Bentuk kumpulan elite untuk generasi berikutnya.

    Jaminan global best:
    Bila skema tidak memakai elitism (keep_global_best=False) -> tidak ada apa pun yang
    disalin, itulah definisi non-elitism.
    Bila elitism aktif, individu terbaik global SELALU ikut dibawa, walaupun
    elite_size diturunkan scheduler. Bila individu tersebut tidak ada di antara elite
    (misalnya karena elite_size sangat kecil), slot elite paling lemah diganti oleh
    global best, sehingga jumlah elite tidak bertambah diam-diam.
    """
    elites = select_elites(scored_population, elite_size)

    if not keep_global_best or global_best is None:
        return elites

    if not elites:
        return [copy.deepcopy(global_best)]

    if all(individual_key(e) != individual_key(global_best) for e in elites):
        elites[-1] = copy.deepcopy(global_best)

    return elites


class EliteScheduler:
    """Scheduler ukuran elitism, teradaptasi dari scheduler laju pembelajaran.

    Prinsipnya sama dengan ReduceLROnPlateau:
      - stagnan selama `patience` generasi -> naikkan tekanan eksploitasi (elitism),
      - diversity turun di bawah lantai      -> turunkan tekanan eksploitasi,
      - setiap perubahan diikuti `cooldown`   -> tidak boleh berosilasi tiap generasi,
      - jumlah elitism selalu dibatasi [minimum, maximum].
    """

    def __init__(
        self,
        initial=INITIAL_ELITISM,
        minimum=MIN_ELITISM,
        maximum=MAX_ELITISM,
        factor=ELITISM_FACTOR,
        patience=SCHEDULER_PATIENCE,
        cooldown=SCHEDULER_COOLDOWN,
        min_diversity=MIN_DIVERSITY,
    ):
        self.size = initial
        self.minimum = minimum
        self.maximum = maximum
        self.factor = factor
        self.patience = patience
        self.cooldown_length = cooldown
        self.min_diversity = min_diversity

        self.stagnation = 0      # generasi sejak global best terakhir membaik
        self.cooldown = 0        # sisa jeda
        self.n_increase = 0
        self.n_decrease = 0

    def _next_up(self):
        """Naikkan secara geometris, minimal +1, dibatasi maksimum."""
        return min(self.maximum, max(self.size + 1, math.ceil(self.size * self.factor)))

    def _next_down(self):
        """Turunkan secara geometris, minimal -1, dibatasi minimum."""
        return max(self.minimum, min(self.size - 1, math.floor(self.size / self.factor)))

    def update(self, improved, diversity_ratio):
        """Dipanggil sekali per generasi. Mengembalikan label aksi atau None."""
        if improved:
            self.stagnation = 0
        else:
            self.stagnation += 1

        # Lantai diversity didahulukan: ini koreksi keselamatan, tidak boleh di-cooldown.
        if diversity_ratio < self.min_diversity and self.size > self.minimum:
            self.size = self._next_down()
            self.cooldown = self.cooldown_length
            self.n_decrease += 1
            return "decrease"

        if self.cooldown > 0:
            self.cooldown -= 1
            return None

        if self.stagnation >= self.patience and self.size < self.maximum:
            self.size = self._next_up()
            self.cooldown = self.cooldown_length
            self.n_increase += 1
            return "increase"

        return None

In [ ]:
# Contoh efek elitism pada populasi awal: berapa individu yang jadi elite
demo_elite_size = STATIC_ELITISM
ranked_initial = sorted(
    [(ind, evaluate(ind, detail=False)) for ind in initial_population],
    key=lambda pair: pair[1]["fitness"],
    reverse=True,
)

rank_table = pd.DataFrame(
    [
        {
            "Rank": i + 1,
            "Fitness": res["fitness"],
            "Penalty": res["penalty"],
            "Status": "ELITE" if i < demo_elite_size else "Non-Elite",
        }
        for i, (ind, res) in enumerate(ranked_initial[:10])
    ]
)
display(rank_table)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(rank_table["Rank"].astype(str), rank_table["Fitness"], color="#4c72b0")
for i, bar in enumerate(bars):
    if i < demo_elite_size:
        bar.set_hatch("//")
        bar.set_color("#2ca02c")
ax.set_title(f"Ranking Populasi Awal: {demo_elite_size} individu terbaik menjadi elite")
ax.set_xlabel("Rank")
ax.set_ylabel("Fitness")
plt.tight_layout()
plt.show()

## 9. Algoritma GA Lengkap

Satu generasi pada eksperimen ini:

```text
1. Evaluasi seluruh individu  -> fitness, penalty
2. Urutkan menurun, catat:
     best fitness, average fitness, diversity, global best
3. Perbarui scheduler elitism  (khusus skema dynamic)
4. Cek kondisi stop:
     - global best tidak membaik selama EARLY_STOPPING_PATIENCE generasi, atau
     - sudah mencapai MAX_GENERATIONS
     (fitness = 100 tidak menghentikan run; hanya dicatat)
5. Bentuk elite (0 / statis / dinamis) + global best
6. Bangun sisa populasi: selection -> crossover -> mutation
7. Generasi baru
```

Dua jenis pencatatan yang perlu dibedakan:

| Kolom | Arti |
|---|---|
| `best_fitness` | fitness tertinggi **pada generasi tersebut** (dapat turun pada non-elitism) |
| `global_best_fitness` | fitness terbaik **yang pernah ditemukan** (monoton naik) |

Early stopping selalu mengacu pada `global_best_fitness`, sama seperti early stopping pada
deep learning yang mengacu pada validation loss terbaik.
`elite_size` yang tercatat pada generasi `t` adalah jumlah elite yang disalin untuk
membentuk generasi `t + 1`.

Satu-satunya kondisi yang menghentikan run adalah **early stopping** (global best tidak
membaik selama `EARLY_STOPPING_PATIENCE` generasi) atau batas `MAX_GENERATIONS`.
Mencapai fitness 100 **tidak** memotong run; nomor generasi saat optimum pertama kali
ditemukan tetap dicatat agar bisa dianalisis tanpa mengubah kurva.

In [ ]:
def run_ga(
    initial_population,
    scheme="dynamic",
    seed=SEED,
    max_generations=MAX_GENERATIONS,
    mutation_rate=MUTATION_RATE,
    static_elitism=STATIC_ELITISM,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    stop_at_optimum=STOP_AT_OPTIMUM,
    **scheduler_options,
):
    """Jalankan GA dengan satu skema elitism.

    scheme = "none" | "static" | "dynamic"
    scheduler_options menerima override konfigurasi scheduler (dipakai saat ablation),
    misalnya maximum=26 atau min_diversity=0.0.
    """
    rng = random.Random(seed)
    population = copy.deepcopy(initial_population)
    pop_size = len(population)

    scheduler = EliteScheduler(**scheduler_options) if scheme == "dynamic" else None
    keep_global_best = scheme != "none"

    if scheme == "none":
        elite_size = 0
    elif scheme == "static":
        elite_size = static_elitism
    else:
        elite_size = scheduler.size

    global_best = None
    global_best_eval = None
    stagnation = 0
    stop_reason = "batas generasi tercapai"
    first_optimum_generation = None

    history = {key: [] for key in [
        "generation", "best_fitness", "avg_fitness", "best_penalty", "diversity",
        "diversity_ratio", "elite_size", "elite_ratio", "stagnation", "cooldown",
        "scheduler_action", "global_best_fitness",
    ]}

    for generation in range(max_generations + 1):
        # ---------- 1. evaluasi & pengurutan ----------
        scored = [(ind, evaluate(ind, detail=False)) for ind in population]
        scored.sort(key=lambda pair: pair[1]["fitness"], reverse=True)

        best_individual, best_eval = scored[0]
        fitness_values = [res["fitness"] for _, res in scored]

        # ---------- 2. global best ----------
        improved = global_best is None or best_eval["fitness"] > global_best_eval["fitness"]
        if improved:
            global_best = copy.deepcopy(best_individual)
            global_best_eval = best_eval
            stagnation = 0
        else:
            stagnation += 1

        if best_eval["fitness"] >= 100 and first_optimum_generation is None:
            first_optimum_generation = generation

        # ---------- 3. scheduler elitism ----------
        diversity = population_diversity(population)
        diversity_ratio = diversity / pop_size

        action = None
        if scheduler is not None:
            action = scheduler.update(improved, diversity_ratio)
            elite_size = scheduler.size
            stagnation = scheduler.stagnation
            cooldown = scheduler.cooldown
        else:
            cooldown = 0

        # ---------- 4. pencatatan ----------
        history["generation"].append(generation)
        history["best_fitness"].append(best_eval["fitness"])
        history["avg_fitness"].append(float(np.mean(fitness_values)))
        history["best_penalty"].append(best_eval["penalty"])
        history["diversity"].append(diversity)
        history["diversity_ratio"].append(diversity_ratio)
        history["elite_size"].append(elite_size)
        history["elite_ratio"].append(elite_size / pop_size)
        history["stagnation"].append(stagnation)
        history["cooldown"].append(cooldown)
        history["scheduler_action"].append(action if action else "-")
        history["global_best_fitness"].append(global_best_eval["fitness"])

        # ---------- 5. kondisi stop ----------
        if stop_at_optimum and best_eval["fitness"] >= 100:
            stop_reason = "optimum (fitness 100) tercapai"
            break
        if stagnation >= early_stopping_patience:
            stop_reason = f"early stopping ({early_stopping_patience} generasi tanpa perbaikan)"
            break
        if generation == max_generations:
            break

        # ---------- 6. elitism ----------
        elites = build_elite_pool(scored, elite_size, global_best, keep_global_best)
        new_population = elites

        # ---------- 7. regenerasi sisa populasi ----------
        while len(new_population) < pop_size:
            parent1 = selection(population, fitness_values, rng=rng)
            parent2 = selection(population, fitness_values, rng=rng)

            child, _ = crossover(parent1, parent2, rng=rng)
            child, _ = mutation(child, mutation_rate=mutation_rate, rng=rng)

            new_population.append(child)

        population = new_population

    return {
        "scheme": scheme,
        "seed": seed,
        "history": pd.DataFrame(history),
        "global_best": global_best,
        "global_best_eval": global_best_eval,
        "final_population": population,
        "stop_reason": stop_reason,
        "generations": len(history["generation"]) - 1,
        "first_optimum_generation": first_optimum_generation,
        "scheduler": scheduler,
    }

## 10. Menjalankan Ketiga Skema

Ketiga skema dijalankan dengan **populasi awal yang sama persis** dan `seed` yang sama.
Variabel bebas satu-satunya adalah skema elitism.

Kriteria pengamatan per generasi:

1. `best_fitness` — solusi terbaik pada generasi tersebut.
2. `avg_fitness` — kualitas rata-rata populasi (indikator eksploitasi).
3. `diversity` — jumlah kromosom unik (indikator eksplorasi).
4. `elite_size` — berapa individu yang dipertahankan oleh elitism.

In [ ]:
def run_scheme(scheme, init_population, seed=SEED):
    return run_ga(init_population, scheme=scheme, seed=seed)


runs = {
    scheme: run_scheme(scheme, initial_population, seed=SEED)
    for scheme in ["none", "static", "dynamic"]
}

histories = {scheme: run["history"] for scheme, run in runs.items()}


def count_fitness_drops(history):
    """Berapa kali best fitness turun dari satu generasi ke generasi berikutnya."""
    return int(np.sum(np.diff(history["best_fitness"].to_numpy()) < 0))


single_summary = pd.DataFrame(
    [
        {
            "Skema": SCHEME_LABELS[scheme],
            "Generasi": run["generations"],
            "Optimum Pertama di Generasi": run["first_optimum_generation"],
            "Best Fitness": run["global_best_eval"]["fitness"],
            "Best Fitness Akhir": hist["best_fitness"].iloc[-1],
            "Avg Fitness Akhir": round(float(hist["avg_fitness"].iloc[-1]), 2),
            "Penurunan Best Fitness": count_fitness_drops(hist),
            "Diversity Akhir": hist["diversity"].iloc[-1],
            "Elite Maks": int(hist["elite_size"].max()),
            "Berhenti Karena": run["stop_reason"],
        }
        for scheme, run in runs.items()
        for hist in [histories[scheme]]
    ]
)

display(single_summary)

## 11. Grafik Ukuran Elitism Dinamis

Grafik ini adalah inti dari eksperimen: bentuk "tangga" elitism pada skema dinamis.

- **naik** (hijau) saat stagnan melewati `SCHEDULER_PATIENCE` → tekanan eksploitasi diperkuat,
- **turun** (merah) saat diversity menyentuh `MIN_DIVERSITY` → tekanan eksplorasi dipulihkan,
- **cooldown** (abu-abu) menjadi alasan scheduler tidak bereaksi pada generasi tertentu.

Kurva elitism berbentuk tangga (*step*), bukan garis mulus, karena scheduler hanya boleh
berubah pada batas cooldown.

In [ ]:
hist_dynamic = histories["dynamic"]

fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(12, 8), sharex=True,
    gridspec_kw={"height_ratios": [1.4, 1]}
)

# --- Panel atas: ukuran elitism dinamis ---
ax1.step(
    hist_dynamic["generation"], hist_dynamic["elite_size"],
    where="post", color=SCHEME_COLORS["dynamic"], linewidth=2,
    label="elite size (digunakan untuk generasi berikutnya)"
)
ax1.axhline(MIN_ELITISM, color="gray", linestyle=":", linewidth=1.2,
            label=f"MIN_ELITISM = {MIN_ELITISM}")
ax1.axhline(MAX_ELITISM, color="firebrick", linestyle="--", linewidth=1.2,
            label=f"MAX_ELITISM = {MAX_ELITISM}")
ax1.axhline(STATIC_ELITISM, color=SCHEME_COLORS["static"], linestyle="-.", linewidth=1.2,
            label=f"Static elitism = {STATIC_ELITISM}")

increase_gens = hist_dynamic.loc[hist_dynamic["scheduler_action"] == "increase", "generation"]
decrease_gens = hist_dynamic.loc[hist_dynamic["scheduler_action"] == "decrease", "generation"]
ax1.scatter(increase_gens, hist_dynamic.loc[hist_dynamic["scheduler_action"] == "increase", "elite_size"],
            color="seagreen", marker="^", s=90, zorder=5, label="kenaikan (stagnan ≥ patience)")
if len(decrease_gens):
    ax1.scatter(decrease_gens, hist_dynamic.loc[hist_dynamic["scheduler_action"] == "decrease", "elite_size"],
                color="crimson", marker="v", s=90, zorder=5, label="penurunan (diversity < lantai)")

ax1.set_title("Perkembangan Ukuran Elitism Dinamis (skema Dynamic Elitism)")
ax1.set_ylabel("elite_size")
ax1.grid(alpha=0.25)
ax1.legend(loc="upper left", fontsize=9)

# --- Panel bawah: diversity sebagai pemicu keputusan scheduler ---
ax2.plot(hist_dynamic["generation"], hist_dynamic["diversity_ratio"],
         color="#9467bd", marker="o", markersize=3, label="rasio diversity")
ax2.axhline(MIN_DIVERSITY, color="crimson", linestyle="--", linewidth=1.2,
            label=f"lantai diversity = {MIN_DIVERSITY:.0%}")
ax2.fill_between(
    hist_dynamic["generation"], 0, hist_dynamic["diversity_ratio"],
    where=hist_dynamic["diversity_ratio"] < MIN_DIVERSITY, color="crimson", alpha=0.15
)
ax2.set_title("Population Diversity sebagai pemicu scheduler")
ax2.set_xlabel("Generasi")
ax2.set_ylabel("rasio individu unik")
ax2.set_ylim(0, 1.05)
ax2.grid(alpha=0.25)
ax2.legend(loc="lower left", fontsize=9)

plt.tight_layout()
plt.show()

scheduler_events = hist_dynamic.loc[hist_dynamic["scheduler_action"] != "-"]
if len(scheduler_events):
    print("Riwayat perubahan elitism:")
    display(scheduler_events[["generation", "best_fitness", "avg_fitness", "diversity_ratio",
                              "elite_size", "scheduler_action"]].reset_index(drop=True))
else:
    print("Tidak ada perubahan elitism pada run ini (elitism konstan di titik awal).")

## 12. Grafik Perkembangan Fitness Ketiga Skema

Dua grafik, masing-masing dengan tiga kurva:

- **Best fitness** — naik monoton pada skema berelitism, dan boleh turun pada non-elitism.
- **Average fitness** — menunjukkan seberapa cepat populasi secara keseluruhan mendekati
  solusi terbaik. Di sinilah dynamic elitism akan terlihat unggul: elitism dinaikkan justru
  ketika populasi berhenti membaik, sehingga rata-rata populasi bergerak lebih cepat.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for scheme, color in SCHEME_COLORS.items():
    hist = histories[scheme]
    label = f"{SCHEME_LABELS[scheme]}"
    axes[0].plot(hist["generation"], hist["best_fitness"], color=color,
                 marker="o", markersize=3, label=label)
    axes[1].plot(hist["generation"], hist["avg_fitness"], color=color, label=label)

axes[0].set_title("Perkembangan Best Fitness")
axes[0].set_xlabel("Generasi")
axes[0].set_ylabel("Best fitness")
axes[0].grid(alpha=0.25)
axes[0].legend()

axes[1].set_title("Perkembangan Average Fitness")
axes[1].set_xlabel("Generasi")
axes[1].set_ylabel("Average fitness")
axes[1].grid(alpha=0.25)
axes[1].legend()

fig.suptitle("Perbandingan Tiga Skema Elitism (seed sama, populasi awal sama)")
plt.tight_layout()
plt.show()

## 13. Grafik Population Diversity Ketiga Skema

`diversity` dihitung sebagai **rasio individu unik terhadap ukuran populasi**
(`len(set(kromosom)) / POP_SIZE`), sehingga nilainya berada pada rentang `0..1` dan
dapat dibandingkan antar skema.

Harapan awal:

- non-elitism: diversity paling tinggi, tetapi tidak pernah menetap pada solusi yang baik,
- static elitism: diversity turun tipis,
- dynamic elitism: diversity turun paling cepat karena elitism dinaikkan berkala.

Kalau kurva dynamic elitism menyentuh lantai `MIN_DIVERSITY`, berarti guardrail bekerja:
scheduler akan menurunkan elitism kembali.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for scheme, color in SCHEME_COLORS.items():
    hist = histories[scheme]
    axes[0].plot(hist["generation"], hist["diversity"], color=color,
                 marker="o", markersize=3, label=SCHEME_LABELS[scheme])
    axes[1].plot(hist["generation"], hist["diversity_ratio"], color=color,
                 marker="o", markersize=3, label=SCHEME_LABELS[scheme])

axes[0].set_title("Jumlah Kromosom Unik")
axes[0].set_ylabel("individu unik")
axes[1].set_title("Rasio Diversity (individu unik / ukuran populasi)")
axes[1].set_ylabel("rasio diversity")
axes[1].set_ylim(0, 1.05)

for ax in axes:
    ax.set_xlabel("Generasi")
    ax.grid(alpha=0.25)
    ax.legend()

axes[1].axhline(MIN_DIVERSITY, color="crimson", linestyle="--", linewidth=1.2,
                label=f"lantai diversity = {MIN_DIVERSITY:.0%}")
axes[1].legend(loc="lower left", fontsize=9)

fig.suptitle("Population Diversity pada Tiga Skema Elitism")
plt.tight_layout()
plt.show()

## 14. Tabel Perkembangan Tiap Generasi

Setiap run menyimpan satu baris per generasi. Tabel di bawah menampilkan:

- tabel lengkap tiap skema (awal dan akhir),
- tabel gabungan yang menyandingkan ketiga skema per generasi,
- tabel khusus riwayat perubahan dari scheduler (kenaikan/penurunan elitism).

In [ ]:
def show_history(title, history):
    """Cetak ringkasan tabel perkembangan satu skema per generasi."""
    columns = [
        "generation", "best_fitness", "avg_fitness", "best_penalty",
        "diversity", "diversity_ratio", "elite_size", "stagnation", "cooldown", "scheduler_action",
    ]
    print(f"--- {title} ---")
    print(f"{len(history)} generasi | best fitness tertinggi = {history['best_fitness'].max()} | "
          f"diversity akhir = {history['diversity_ratio'].iloc[-1]:.2f}")
    display(history[columns].round(3).reset_index(drop=True))


for scheme in ["none", "static", "dynamic"]:
    show_history(SCHEME_LABELS[scheme], histories[scheme])
    print()

In [ ]:
# Tabel gabungan: tiga skema berdampingan pada generasi yang sama
max_gen = max(len(hist) for hist in histories.values()) - 1
combined_history = pd.DataFrame({"generation": np.arange(max_gen + 1)})

for scheme in ["none", "static", "dynamic"]:
    hist = histories[scheme].set_index("generation")
    label = SCHEME_LABELS[scheme]
    for column, tag in [
        ("best_fitness", "best"),
        ("avg_fitness", "avg"),
        ("diversity_ratio", "diversity"),
        ("elite_size", "elite"),
    ]:
        combined_history[f"{label} - {tag}"] = hist[column].reindex(
            combined_history["generation"]
        ).to_numpy()

if SHOW_FULL_TABLE:
    display(combined_history.round(3).reset_index(drop=True))
else:
    display(combined_history.round(3).head(20).reset_index(drop=True))
    print(f"... total {max_gen + 1} generasi; tampilkan 20 baris pertama "
          f"(set SHOW_FULL_TABLE = True untuk melihat seluruh tabel)")

print("\n10 generasi terakhir:")
display(combined_history.round(3).tail(10).reset_index(drop=True))

## 15. Analisis: Apakah Elitism Lebih Besar = Diversity Lebih Rendah?

Bagian ini menguji kekhawatiran utama eksperimen: **apakah menaikkan elitism selalu
menurunkan diversity, dan seberapa besar pengaruhnya?**

Dua cara diukur:

1. Korelasi Pearson antara `elite_size` dan `diversity_ratio` di dalam run dynamic.
2. Regresi linier sederhana: diversity sebagai fungsi elite_size.

Nilai korelasi negatif yang kuat berarti pengaman diversity **wajib** ada, karena scheduler
tidak boleh menaikkan elitism tanpa memantau diversity.

In [ ]:
hist_dynamic = histories["dynamic"]
subset = hist_dynamic[["elite_size", "diversity_ratio", "diversity", "avg_fitness", "best_fitness"]]

correlation = subset["elite_size"].corr(subset["diversity_ratio"])
slope, intercept = np.polyfit(subset["elite_size"], subset["diversity_ratio"], 1)

print(f"Korelasi Pearson (elite_size vs rasio diversity) : r = {correlation:.3f}")
print(f"Regresi: rasio diversity ≈ {slope:.4f} × elite_size + {intercept:.4f}")
print(f"Artinya: setiap elitism +1 memotong diversity sekitar {abs(slope) * 100:.1f} poin persen")

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.scatter(subset["elite_size"], subset["diversity_ratio"], alpha=0.65,
           color="#2ca02c", label="titik generasi")
x_fit = np.linspace(subset["elite_size"].min(), subset["elite_size"].max(), 50)
ax.plot(x_fit, slope * x_fit + intercept, color="firebrick", linestyle="--",
        label=f"regresi (r = {correlation:.2f})")
ax.set_xlabel("elite_size pada generasi tersebut")
ax.set_ylabel("rasio diversity")
ax.set_title("Hubungan Ukuran Elitism dengan Population Diversity (run Dynamic Elitism)")
ax.set_ylim(0, 1.05)
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

## 16. Eksperimen Multi-Run

GA bersifat stokastik; satu kali run tidak cukup untuk menyimpulkan perilaku.
Karena itu ketiga skema dijalankan ulang dengan `N_RUNS` seed berbeda.

Pada setiap run, **populasi awal dibuat dari seed tersebut** dan dipakai bersama oleh ketiga
skema, sehingga perbandingannya tetap terkendali.

Statistik yang dikumpulkan:

| Metrik | Arti |
|---|---|
| `Best Fitness` | fitness terbaik yang pernah ditemukan |
| `Fitness Akhir` | best fitness pada generasi terakhir (bisa lebih rendah dari yang pernah dicapai) |
| `Penurunan Best Fitness` | banyaknya kali kurva best fitness turun |
| `Rata-rata Fitness` | kualitas rata-rata populasi sepanjang run |
| `Diversity Min` | titik terendah diversity (indikasi risiko premature convergence) |
| `Elite Maks` | elitism tertinggi yang pernah dipakai |

In [ ]:
multi_histories = {}
multi_rows = []

for seed in range(N_RUNS):
    rng_init = random.Random(seed)
    init_population = [create_individual(rng_init) for _ in range(POP_SIZE)]

    for scheme in ["none", "static", "dynamic"]:
        result = run_ga(init_population, scheme=scheme, seed=seed)
        hist = result["history"]
        multi_histories[(scheme, seed)] = hist

        scheduler = result["scheduler"]
        multi_rows.append({
            "Seed": seed,
            "Skema": SCHEME_LABELS[scheme],
            "Generasi": result["generations"],
            "Optimum Pertama": result["first_optimum_generation"],
            "Best Fitness": result["global_best_eval"]["fitness"],
            "Fitness Akhir": hist["best_fitness"].iloc[-1],
            "Rata-rata Fitness": round(float(hist["avg_fitness"].mean()), 2),
            "Penurunan Best Fitness": count_fitness_drops(hist),
            "Diversity Akhir": hist["diversity_ratio"].iloc[-1],
            "Diversity Min": hist["diversity_ratio"].min(),
            "Elite Maks": int(hist["elite_size"].max()),
            "Naik Elitism": scheduler.n_increase if scheduler else 0,
            "Turun Elitism": scheduler.n_decrease if scheduler else 0,
            "Berhenti": result["stop_reason"],
        })

df_multi = pd.DataFrame(multi_rows)
display(df_multi)

In [ ]:
multi_summary = (
    df_multi
    .groupby("Skema")
    .agg(
        Rata_rata_Best=("Best Fitness", "mean"),
        Std_Best=("Best Fitness", "std"),
        Rata_rata_Fitness_Akhir=("Fitness Akhir", "mean"),
        Rata_rata_Avg_Fitness=("Rata-rata Fitness", "mean"),
        Total_Penurunan=("Penurunan Best Fitness", "mean"),
        Rata_rata_Diversity_Akhir=("Diversity Akhir", "mean"),
        Diversity_Terendah=("Diversity Min", "mean"),
        Elite_Maks=("Elite Maks", "mean"),
        Naik_Elitism=("Naik Elitism", "mean"),
        Turun_Elitism=("Turun Elitism", "mean"),
        Rata_rata_Generasi=("Generasi", "mean"),
    )
    .reindex([SCHEME_LABELS["none"], SCHEME_LABELS["static"], SCHEME_LABELS["dynamic"]])
    .round(2)
)
display(multi_summary)

# Berapa banyak run yang mencapai solusi tanpa pelanggaran (fitness 100)?
optimum_table = (
    df_multi.assign(Solus_Optimal=df_multi["Best Fitness"] >= 100)
    .groupby("Skema")["Solus_Optimal"]
    .agg(["sum", "count", "mean"])
    .reindex([SCHEME_LABELS["none"], SCHEME_LABELS["static"], SCHEME_LABELS["dynamic"]])
    .round(2)
)
optimum_table.columns = ["Run mencapai fitness 100", "Jumlah run", "Persentase"]
display(optimum_table)

# Kapan optimum pertama kali ditemukan? Rata-rata hanya dari run yang mencapainya,
# karena run yang tidak pernah sampai ke 100 berarti "tidak pernah tercapai".
first_optimum_table = (
    df_multi.dropna(subset=["Optimum Pertama"])
    .groupby("Skema")["Optimum Pertama"]
    .agg(["count", "mean", "min", "max"])
    .reindex([SCHEME_LABELS["none"], SCHEME_LABELS["static"], SCHEME_LABELS["dynamic"]])
    .round(1)
)
first_optimum_table.columns = [
    "Run yang mencapai optimum", "Rata-rata generasi optimum",
    "Paling cepat", "Paling lambat",
]
display(first_optimum_table)

In [ ]:
# Kurva rata-rata multi-run (mean ± std antar run)
max_generation = max(len(hist) for hist in multi_histories.values()) - 1
generations = np.arange(max_generation + 1)


def aggregate_column(scheme, column):
    """Rata-rata dan std sebuah kolom pada seluruh run untuk satu skema.

    Run yang berhenti lebih awal diisi NaN, sehingga rata-rata dihitung hanya
    dari run yang masih berjalan pada generasi tersebut.
    """
    padded_rows = []
    for seed in range(N_RUNS):
        values = multi_histories[(scheme, seed)][column].to_numpy()
        padded = np.full(max_generation + 1, np.nan)
        padded[:len(values)] = values
        padded_rows.append(padded)

    frame = pd.DataFrame(np.array(padded_rows).T, columns=[f"seed_{s}" for s in range(N_RUNS)])
    return frame.mean(axis=1), frame.std(axis=1), frame.notna().sum(axis=1)


fig, axes = plt.subplots(2, 2, figsize=(14, 9))

plots = [
    (axes[0, 0], "best_fitness", "Best Fitness", "Multi-run: rata-rata Best Fitness"),
    (axes[0, 1], "avg_fitness", "Average Fitness", "Multi-run: rata-rata Average Fitness"),
    (axes[1, 0], "diversity", "Jumlah Kromosom Unik", "Multi-run: rata-rata Diversity"),
    (axes[1, 1], "elite_size", "elite_size", "Multi-run: rata-rata elite_size"),
]

for ax, column, ylabel, title in plots:
    for scheme, color in SCHEME_COLORS.items():
        mean, std, active = aggregate_column(scheme, column)
        mean = mean.to_numpy()
        std = std.fillna(0).to_numpy()
        ax.plot(generations, mean, color=color, label=SCHEME_LABELS[scheme])
        ax.fill_between(generations, mean - std, mean + std, color=color, alpha=0.15)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Generasi")
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=9)

fig.suptitle(f"Ringkasan {N_RUNS} Run per Skema (bayang = ±1 standar deviasi)")
plt.tight_layout()
plt.show()

print("Panjang run per skema (garis rata-rata di atas hanya dihitung dari run yang belum berhenti):")
run_length_table = pd.DataFrame(
    [
        {
            "Skema": SCHEME_LABELS[scheme],
            "Run terpendek": min(len(multi_histories[(scheme, s)]) for s in range(N_RUNS)),
            "Run terpanjang": max(len(multi_histories[(scheme, s)]) for s in range(N_RUNS)),
            "Jumlah generasi aktif di generasi terakhir": int(aggregate_column(scheme, "best_fitness")[2].iloc[-1]),
        }
        for scheme in SCHEME_COLORS
    ]
)
display(run_length_table)

## 17. Ablasi: Efektivitas Pengaman Scheduler

Bagian ini menjawab pertanyaan "apakah dynamic elitism berbahaya?" secara langsung:
bandingkan scheduler yang **aman** (ada plafon, lantai diversity, cooldown) dengan scheduler
yang **dibiarkan lepas** (batas atas tinggi, tanpa lantai diversity).

Tiga varian:

| Varian | `MAX_ELITISM` | `MIN_DIVERSITY` | Makna |
|---|---:|---:|---|
| A — Dinamis (aman) | 12 | 0.60 | konfigurasi utama notebook |
| B — Tanpa lantai diversity | 12 | 0.00 | elitism boleh naik terus |
| C — Tanpa batas atas | 26 | 0.00 | ~87% populasi bisa jadi elite |

Jika kekhawatiran tersebut benar, maka pada varian C kita harus melihat:
diversity turun paling dalam, `Elite Maks` paling tinggi, dan best fitness paling buruk.

In [ ]:
variants = {
    # konfigurasi utama notebook
    "A - Dinamis (aman)": {},
    # lantai diversity dimatikan: elitism boleh naik tanpa umpan balik
    "B - Tanpa lantai diversity": {"min_diversity": 0.0},
    # lantai diversity dimatikan DAN batas atas dinaikkan menjadi ~87% populasi
    "C - Tanpa batas atas": {"maximum": 26, "minimum": 0, "min_diversity": 0.0},
}

ablation_rows = []
ablation_histories = {}

for seed in range(N_RUNS):
    rng_init = random.Random(seed)
    init_population = [create_individual(rng_init) for _ in range(POP_SIZE)]

    for label, options in variants.items():
        result = run_ga(init_population, scheme="dynamic", seed=seed, **options)
        hist = result["history"]
        scheduler = result["scheduler"]
        ablation_histories[(label, seed)] = hist

        ablation_rows.append({
            "Seed": seed,
            "Varian": label,
            "Generasi": result["generations"],
            "Best Fitness": result["global_best_eval"]["fitness"],
            "Rata-rata Fitness": round(float(hist["avg_fitness"].mean()), 2),
            "Diversity Akhir": hist["diversity_ratio"].iloc[-1],
            "Diversity Min": hist["diversity_ratio"].min(),
            "Elite Maks": int(hist["elite_size"].max()),
            "Elite Rasio Maks": round(float(hist["elite_ratio"].max()), 2),
            "Naik Elitism": scheduler.n_increase,
            "Turun Elitism": scheduler.n_decrease,
            "Berhenti": result["stop_reason"],
        })

df_ablation = pd.DataFrame(ablation_rows)
display(df_ablation.head(15))

ablation_summary = (
    df_ablation
    .groupby("Varian")
    .agg(
        Rata_rata_Best=("Best Fitness", "mean"),
        Rata_rata_Avg_Fitness=("Rata-rata Fitness", "mean"),
        Diversity_Akhir=("Diversity Akhir", "mean"),
        Diversity_Terendah=("Diversity Min", "mean"),
        Elite_Maks=("Elite Maks", "mean"),
        Elite_Rasio_Maks=("Elite Rasio Maks", "mean"),
        Naik_Elitism=("Naik Elitism", "mean"),
        Turun_Elitism=("Turun Elitism", "mean"),
        Rata_rata_Generasi=("Generasi", "mean"),
    )
    .reindex(list(variants))
    .round(2)
)
display(ablation_summary)

In [ ]:
# Grafik ablasi: rata-rata multi-run untuk ketiga varian scheduler
ablation_max_generation = max(len(hist) for hist in ablation_histories.values()) - 1
ablation_generations = np.arange(ablation_max_generation + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
variant_colors = ["#2ca02c", "#ff7f0e", "#d62728"]

for color, label in zip(variant_colors, variants):
    frames_diversity, frames_best = [], []
    for seed in range(N_RUNS):
        hist = ablation_histories[(label, seed)]

        padded = np.full(ablation_max_generation + 1, np.nan)
        values = hist["diversity_ratio"].to_numpy()
        padded[:len(values)] = values
        frames_diversity.append(padded)

        padded = np.full(ablation_max_generation + 1, np.nan)
        values = hist["best_fitness"].to_numpy()
        padded[:len(values)] = values
        frames_best.append(padded)

    mean_div = pd.DataFrame(np.array(frames_diversity).T).mean(axis=1).to_numpy()
    mean_best = pd.DataFrame(np.array(frames_best).T).mean(axis=1).to_numpy()
    axes[0].plot(ablation_generations, mean_div, color=color, label=label)
    axes[1].plot(ablation_generations, mean_best, color=color, label=label)

axes[0].axhline(MIN_DIVERSITY, color="crimson", linestyle="--", linewidth=1.2,
                label=f"lantai diversity = {MIN_DIVERSITY:.0%}")
axes[0].set_title("Rasio Diversity pada Tiga Varian Scheduler")
axes[0].set_ylabel("rasio diversity")
axes[0].set_ylim(0, 1.05)
axes[1].set_title("Best Fitness pada Tiga Varian Scheduler")
axes[1].set_ylabel("best fitness")

for ax in axes:
    ax.set_xlabel("Generasi")
    ax.grid(alpha=0.25)
    ax.legend(fontsize=9)

fig.suptitle("Ablasi: Efektivitas Pengaman Scheduler (mean over run)")
plt.tight_layout()
plt.show()

## 18. Jadwal Terbaik yang Ditemukan

Bagian ini menutup eksperimen dengan hasil: kromosom terbaik diubah menjadi jadwal
yang dapat dibaca manusia. Jadwal diambil dari skema dynamic elitism pada seed utama.

In [ ]:
best_schedule = runs["dynamic"]["global_best"]
best_eval = evaluate(best_schedule)


def schedule_to_dataframe(individual):
    rows = []
    for course, gene in zip(courses, individual):
        slot, room = gene
        rows.append({
            "Mata Kuliah": course["name"],
            "Dosen": course["lecturer"],
            "Semester": course["semester"],
            "Mahasiswa": course["students"],
            "Waktu": timeslots[slot],
            "Ruang": rooms[room]["name"],
            "Kapasitas": rooms[room]["capacity"],
        })
    return pd.DataFrame(rows).sort_values(["Waktu", "Ruang"]).reset_index(drop=True)


df_best_schedule = schedule_to_dataframe(best_schedule)

print("BEST FITNESS :", best_eval["fitness"])
print("TOTAL PENALTY:", best_eval["penalty"])
display(df_best_schedule)

In [ ]:
# Visualisasi jadwal sebagai matriks slot x ruang
schedule_matrix = [["" for _ in rooms] for _ in timeslots]

for course, gene in zip(courses, best_schedule):
    slot, room = gene
    label = course["name"]
    if schedule_matrix[slot][room]:
        schedule_matrix[slot][room] += "\n" + label
    else:
        schedule_matrix[slot][room] = label

fig, ax = plt.subplots(figsize=(11, 6))
ax.set_xlim(0, len(rooms))
ax.set_ylim(0, len(timeslots))
ax.invert_yaxis()

ax.set_xticks(np.arange(len(rooms)) + 0.5)
ax.set_xticklabels([f"{r['name']}\nCap {r['capacity']}" for r in rooms])
ax.set_yticks(np.arange(len(timeslots)) + 0.5)
ax.set_yticklabels(timeslots)

for i in range(len(timeslots)):
    for j in range(len(rooms)):
        ax.add_patch(plt.Rectangle((j, i), 1, 1, fill=False))
        text = schedule_matrix[i][j] if schedule_matrix[i][j] else "-"
        ax.text(j + 0.5, i + 0.5, text, ha="center", va="center", fontsize=9)

ax.set_title("Visualisasi Jadwal Terbaik (Dynamic Elitism)")
ax.set_xlabel("Ruang")
ax.set_ylabel("Waktu")
plt.tight_layout()
plt.show()

In [ ]:
# Laporan pelanggaran jadwal terbaik
if best_eval["violations"]:
    print(f"Masih ada {len(best_eval['violations'])} pelanggaran:")
    for i, violation in enumerate(best_eval["violations"], 1):
        print(f"{i}. {violation}")
else:
    print("Tidak ada pelanggaran hard maupun soft constraint pada jadwal terbaik.")

if best_eval["breakdown"]:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(list(best_eval["breakdown"].keys()), list(best_eval["breakdown"].values()), color="#c44e52")
    ax.set_title("Breakdown Penalti Jadwal Terbaik")
    ax.set_ylabel("Total penalti")
    ax.tick_params(axis="x", rotation=20)
    plt.tight_layout()
    plt.show()

## 19. Kesimpulan

### 19.1 Peran elitism

Elitism menjamin satu hal yang tidak dapat dijamin oleh operator lain:

$$
BestFitness_{t+1} \ge BestFitness_{t}
$$

karena individu terbaik tidak pernah dirombak oleh crossover, mutation, maupun penggantian
seluruh populasi. Pada eksperimen multi-run, kurva best fitness pada kedua skema berelitism
tidak pernah turun, sedangkan non-elitism menurunkannya berkali-kali.

### 19.2 Static vs Dynamic

- **Static elitism** = tekanan eksploitasi konstan. Stabil dan diversity-nya tinggi,
  tetapi lambat keluar dari *stagnation*.
- **Dynamic elitism** = tekanan eksploitasi yang meningkat hanya saat perlu
  (stagnan ≥ patience) dan menurun kembali bila diversity memburuk. Rata-rata fitness naik
  lebih cepat, tetapi diversity-nya lebih rendah. Ia menukar *eksplorasi* untuk
  *eksploitasi*, secara sadar dan terukur.

### 19.3 Apakah dynamic elitism berbahaya?

**Berbahaya hanya jika** salah satu kondisi berikut tidak dijaga:

- `MAX_ELITISM` tidak dibatasi, sehingga elite tumbuh sampai hampir seluruh populasi,
- tidak ada lantai diversity, sehingga tidak ada umpan balik yang menghentikan kenaikan,
- scheduler bereaksi setiap generasi, sehingga elitism naik-turun tidak karuan (osilasi),
- global best tidak disimpan eksplisit, sehingga penurunan elitism bisa menghapus solusi terbaik.

Dengan pengaman yang dipakai pada notebook ini, risiko tersebut terkendali: elitism berhenti
di `MAX_ELITISM`, diturunkan kembali saat diversity menyentuh lantai, beristirahat selama
cooldown, dan global best selalu dibawa walau elitism diturunkan. Hasil ablasi pada section 17
menunjukkan konsekuensinya secara kuantitatif: tanpa pengaman, diversity anjlok dan best
fitness justru memburuk.

### 19.4 Trade-off inti

$$
\underbrace{\text{elitism besar}}_{\text{eksploitasi}} \quad\Longleftrightarrow\quad
\underbrace{\text{diversity besar}}_{\text{eksplorasi}}
$$

Elitism bukan tombol "selalu lebih baik". Ia adalah cara membeli kecepatan konvergensi
dengan mengurangi eksplorasi. Pada kebanyakan masalah, harga yang tepat adalah
**elitism secukupnya yang diatur scheduler dan dijaga diversity**, bukan elitism maksimal.

---

### Ide pengembangan lanjutan

- Jadikan scheduler dua arah penuh: naikkan saat stagnan, turunkan saat stagnan **dan** diversity tinggi.
- Ganti diversity berbasis jumlah individu unik dengan metrik jarak antarkromosom
  (misalnya jarak Hamming rata-rata atau jarak nilai fitness).
- Bandingkan dengan varian *aging* / *rejuvenation* dan dengan random immigrants.
- Uji sensitivitas `ELITISM_FACTOR`, `MAX_ELITISM`, `MIN_DIVERSITY`, dan `SCHEDULER_PATIENCE`.
- Uji pada instance yang lebih besar agar ruang solusi tidak mudah dijangkau optimum.

## 20. Pertanyaan Diskusi

1. Mengapa best fitness pada GA dengan elitism tidak pernah turun, sementara pada non-elitism bisa?
2. Mengapa dynamic elitism bisa lebih cepat keluar dari *stagnation* dibanding static elitism?
3. Apa gunanya *cooldown* pada scheduler? Apa yang terjadi tanpa *cooldown*?
4. Mengapa lantai diversity (`MIN_DIVERSITY`) harus menang prioritas atas kenaikan elitism?
5. Mengapa global best tetap perlu disimpan eksplisit jika elitism sudah aktif?
6. Apakah elitism yang besar selalu memperpendek waktu menuju solusi optimal? Buktikan dari data.
7. Bagaimana hasil eksperimen akan berubah jika `POP_SIZE` diperkecil menjadi 10?
8. Jika instance penjadwalan dibuat jauh lebih besar (30 mata kuliah), apakah kesimpulan
   "dynamic elitism lebih cepat" masih berlaku?
9. Bagaimana cara memastikan kesimpulan ini bukan artefak dari satu seed?
10. Metrik diversity apa yang lebih tepat jika dua individu berbeda tetapi berada di
    wilayah solusi yang sama?